# M46 lab solution: design document for Capstone A

**Brief.** Architecture review: write a two-page design document for your capstone that justifies each component against a simpler alternative.

This is the worked document for Capstone A, Enterprise Knowledge Assistant (`capstones/A_enterprise_knowledge_assistant/README.md`): *a cited Q&A assistant over a company's PDFs, wiki pages and tables.* The capstone page lists its required components with v1 module numbers; this document uses the v2 numbers (M11 ingestion, M12 chunking, M13 embeddings, M14 vector stores, M16 hybrid retrieval, M18 reranking, M19 end-to-end RAG and citations, M29 CRAG, and the M10 baseline).

The Larkspur Cloud handbook in `../data/larkspur/` stands in for the company's documents. A code cell appears only where a row of the table makes a claim that the golden set can check. Each check prints its numbers; the table cells that depend on them read *pending run* until the notebook is run.

Run with the default environment (`uv sync`). Sections 1, 2 and 4 run offline. Section 3 embeds the handbook and needs `DATABRICKS_HOST` and `DATABRICKS_TOKEN` (the default embeddings of `src/helpers`).

## The design

```mermaid
flowchart LR
    subgraph Index["Index time"]
        S["PDFs, wiki pages, tables"] --> L["Loaders and metadata schema<br/>M11"]
        L --> C["Chunker<br/>M12"]
        C --> E["Embeddings<br/>M13"]
        C --> K["Keyword index"]
        E --> V[("pgvector, incremental<br/>M14")]
    end
    subgraph Query["Query time"]
        Q["Question"] --> H["Hybrid retrieval<br/>M16"]
        V --> H
        K --> H
        H --> R["Reranker<br/>M18"]
        R --> G{"Relevance check<br/>M29"}
        G -- relevant --> A["Answer with per-claim citations<br/>M19"]
        G -- nothing relevant --> D["Decline: the documents do not say"]
    end
```

The model that writes the answer is a prompted, hosted model reading retrieved chunks. Whether a fine-tuned model replaces or joins it is the last row of the table, and it waits on the M45 verdict.

## Components against simpler alternatives

| Component | Simpler alternative | Why the simpler one is not enough | Check | Model calls added per question |
|---|---|---|---|---|
| Plain RAG (M10): recursive split, TF-IDF top 4, one prompted call | None; this is the baseline every row is measured against | *Pending run*: section 1 shows which golden questions it misses and of which type | Section 1, `lab_kit.baseline` | 1 |
| Loaders and a metadata schema (M11) | One text loader, no metadata | The capstone corpus mixes PDFs, wiki pages and tables. Citations need a source and a page or section per chunk, and incremental indexing needs a document id and a modified time | None here: the Larkspur handbook is one format. The capstone's ingestion tests cover it | 0 |
| Chunker with a measured size (M12) | The baseline's 500-character recursive split | *Pending run*: section 2 compares 250, 500 and 1,000 characters on the golden set | Section 2 | 0 |
| Dense embeddings (M13) | TF-IDF alone | A customer writes "goes down" where the handbook says "outage"; keyword overlap misses that, as the M46 decision framework page shows for TF-IDF. *Pending run*: section 3 | Section 3 | 0 (embedding calls, not model calls) |
| pgvector with incremental indexing (M14) | An in-memory index rebuilt on every change | A rebuild re-embeds every chunk when one document changes, and an in-memory index does not survive a restart. Policies change, and the M45 fact-change test is the case for cheap updates | None here: needs Docker. The capstone measures re-index time for one changed document | 0 |
| Hybrid retrieval (M16) | Dense retrieval alone | Exact tokens such as `429`, `AES-256` or a plan name carry the answer and an embedding can blur them. *Pending run*: section 3 compares dense, TF-IDF and their fusion | Section 3 | 0 |
| Reranker (M18) | The fused order from hybrid retrieval | *Pending run*. Keep it only if MRR on the golden set rises; a cross-encoder adds latency but no model call | Not in this notebook: needs the `hf` group. The capstone evaluation reports MRR with and without it | 0 (a local cross-encoder) |
| Relevance check with a decline branch (M29) | A score threshold on the top retrieved chunk | *Pending run*: section 4 tests whether any threshold separates the five out-of-corpus questions from the answerable ones. CRAG's web-search branch is replaced by a decline, because the company's facts live in private documents the web cannot see (the M29 point quoted on the M46 decision framework page) | Section 4 | 1 if the retrieved chunks are graded as one block; k (here 4) if each chunk is graded, the trade-off M28 describes |
| Answer with per-claim citations (M19) | A list of sources under the answer | A multi-hop answer such as `g02` or `g17` joins sentences from two handbook files. A source list cannot show which claim comes from which file, so a reviewer cannot check one claim without reading every source | None here: needs model calls. The capstone evaluation checks each cited claim against its chunk | 0 (part of the answer call) |
| RAG or fine-tune (M45) | Fine-tune a small model on the documents and skip retrieval | **Depends on the M45 verdict, pending run.** The M45 lab compares the M10 baseline with a fine-tuned `flan-t5-small` on refund and billing questions and changes one fact afterwards. If the fine-tuned model keeps the old fact, documents that change must stay in retrieval, and fine-tuning is left for behaviour (house style, refusals) that a prompt fails to produce | M45 lab (`p13_design_decisions/m45_fine_tuning_model_adaptation/lab/`) | 0 if retrieval stays; a fine-tuned model would replace the answer call, not add one |

## Left out, and the evidence that would bring each back

The M46 decision chain reaches routing, a different index shape, an agent loop or a check only when the golden set shows the failure each one fixes.

- **Query decomposition or routing (M17, M30):** add only if section 1 shows `multi_hop` questions missing evidence that the single-hop ones find.
- **Summary trees or a graph (M42 to M44):** add only if the capstone's golden set has theme or relationship questions the chunk index cannot answer. The Larkspur set has none of either kind.
- **An agent loop (M23, M25, M31):** add only if a question's next search depends on the last result. No golden question needs that.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer

load_dotenv()
sys.path.insert(0, str(Path("../../../../src").resolve()))
from lab_kit import baseline, hit_rate, is_hit, load_handbook, load_jsonl, mrr, recall_at_k, tfidf_retriever

CORPUS = Path("../data/larkspur")
K = 4
golden = load_jsonl(CORPUS / "golden.jsonl")
pages = load_handbook(CORPUS / "handbook")


def chunks_of(pages, size=500, overlap=50):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    return [c for p in pages for c in splitter.split_text(p["text"])]


def score(results):
    return {"hit_rate": round(hit_rate(results, golden), 2), "recall_at_k": round(recall_at_k(results, golden), 2),
            "mrr": round(mrr(results, golden), 2)}


chunks = chunks_of(pages)
print(f"{len(pages)} handbook pages, {len(chunks)} chunks, {len(golden)} golden questions")

## 1. Is plain RAG enough?

`lab_kit.baseline` is the M10 baseline: the same split and TF-IDF top 4 the table's first row describes, scored on the 25 golden questions with evidence. The second cell lists each miss with its question type, which decides whether decomposition or routing earns a place.

In [ ]:
print(baseline(CORPUS, k=K))

In [ ]:
search = tfidf_retriever(chunks, K)
misses = [{"id": g["id"], "type": g["type"], "question": g["question"]}
          for g in golden if g["evidence_spans"] and not any(is_hit(c, g["evidence_spans"]) for c in search(g["question"]))]
print(f"{len(misses)} of {sum(bool(g['evidence_spans']) for g in golden)} answerable questions have no evidence in the top {K}")
pd.DataFrame(misses)

## 2. Chunk size

The baseline's 500 characters against half and double, each with a 10% overlap, all with TF-IDF top 4.

In [ ]:
chunk_rows = []
for size in (250, 500, 1000):
    r = baseline(CORPUS, k=K, chunk_size=size, chunk_overlap=size // 10)
    chunk_rows.append({"chunk_size": size, "chunks": len(chunks_of(pages, size, size // 10)), **{m: round(r[m], 2) for m in ("hit_rate", "recall_at_k", "mrr")}})
pd.DataFrame(chunk_rows)

## 3. TF-IDF, dense and hybrid retrieval

Dense retrieval embeds every chunk with the course's default embeddings and ranks by cosine similarity. Hybrid fuses the TF-IDF and dense rankings with reciprocal rank fusion, `1 / (60 + rank)`, the constant M16 cites from `EnsembleRetriever`. All three return a top 4 from the same 500-character chunks.

In [ ]:
from helpers import get_embeddings

embeddings = get_embeddings()
chunk_vecs = np.array(embeddings.embed_documents(chunks))
chunk_vecs /= np.linalg.norm(chunk_vecs, axis=1, keepdims=True)
tfidf_rank = tfidf_retriever(chunks, len(chunks))


def dense_order(question):
    q = np.array(embeddings.embed_query(question))
    return list(np.argsort(-(chunk_vecs @ (q / np.linalg.norm(q)))))


def hybrid_order(question, rrf_k=60):
    fused = {}
    for rank, c in enumerate(tfidf_rank(question), 1):
        i = chunks.index(c)
        fused[i] = fused.get(i, 0) + 1 / (rrf_k + rank)
    for rank, i in enumerate(dense_order(question), 1):
        fused[i] = fused.get(i, 0) + 1 / (rrf_k + rank)
    return sorted(fused, key=lambda i: -fused[i])


retrievers = {
    "tf-idf": lambda q: search(q),
    "dense": lambda q: [chunks[i] for i in dense_order(q)[:K]],
    "hybrid (RRF)": lambda q: [chunks[i] for i in hybrid_order(q)[:K]],
}
retrieval_rows = [{"retriever": name, **score({g["id"]: fn(g["question"]) for g in golden})} for name, fn in retrievers.items()]
pd.DataFrame(retrieval_rows)

## 4. Can a score threshold catch out-of-corpus questions?

The simpler alternative to a relevance check is a cut-off on the top retrieval score. It works only if every out-of-corpus question scores below every answerable one. The cell prints the top TF-IDF cosine score per question, sorted, and whether such a threshold exists.

In [ ]:
vec = TfidfVectorizer(stop_words="english")
matrix = vec.fit_transform(chunks)
top = pd.DataFrame([{"id": g["id"], "type": g["type"], "top_score": round(float((matrix @ vec.transform([g["question"]]).T).max()), 3)}
                    for g in golden]).sort_values("top_score")
ooc = top[top["type"] == "out_of_corpus"]["top_score"]
answerable = top[top["type"] != "out_of_corpus"]["top_score"]
print(f"highest out-of-corpus score {ooc.max():.3f}, lowest answerable score {answerable.min():.3f}, "
      f"a threshold separates them: {ooc.max() < answerable.min()}")
top

## Results

*Pending run.* Fill the table rows marked *pending run* from these outputs, with the date and the embedding model, and record each decision below.

In [ ]:
print("Chunk size:")
print(pd.DataFrame(chunk_rows).to_string(index=False))
print("\nRetrievers at k =", K)
print(pd.DataFrame(retrieval_rows).to_string(index=False))
print(f"\nBaseline misses: {len(misses)}, of which multi_hop: {sum(m['type'] == 'multi_hop' for m in misses)}")
print(f"Score threshold separates out-of-corpus questions: {ooc.max() < answerable.min()}")

## Decisions

*Pending run.* One line per row, each naming the number that decided it:

- **Plain RAG enough?** *Pending run.* If section 1 shows no misses, ship the baseline plus the decline branch and log the misses.
- **Chunk size:** *pending run.* Keep 500 unless another size raises recall@4 on the golden set.
- **Dense and hybrid:** *pending run.* Keep hybrid only if it beats both TF-IDF and dense alone; keep dense if it beats TF-IDF.
- **Relevance check:** *pending run.* If a threshold separates the out-of-corpus questions, the threshold is the decline branch at 0 model calls; otherwise add the M29 grader.
- **RAG or fine-tune:** *pending the M45 verdict.*